# Polytope Lib Tests

In [1]:
from pycvxset import Polytope
import numpy as np
import matplotlib.pyplot as plt

import os
import mosek

os.environ["MOSEKLM_LICENSE_FILE"] = "27007@192.168.100.118"
mosek.Env().getversion()
mosek.Env().putlicensepath("27007@192.168.100.118")
mosek.Env().expirylicenses()

-1

In [2]:
from itertools import product
def cartesian_product(*polytopes):
    V_rep = [p.V for p in polytopes]
    Combinations = []
    for veritces in product(*V_rep):
        Combinations.append(np.concatenate(veritces))
    return Polytope(V=np.array(Combinations))

In [3]:
# Define the polytope using H-representation with 2D

A2 = np.array([[1, 0], [0, 1], [-1, 0], [0, -1]])
b2 = np.array([1, 2, 1, 2])

P_2D = Polytope(A=A2, b=b2)

# Define the polytope using H-representation with 1D

P_1D = Polytope(A=np.array([[1], [-1]]), b=np.array([3, 3]))

# Define the polytope using H-representation with 3D as Chartesian product of 1D and 2D

P = cartesian_product(P_1D, P_2D)


ax, _, _ = P.plot()
plt.show() 

KeyboardInterrupt: 

In [ ]:
F = Polytope(A = P.A/P.b[:,np.newaxis], b= np.ones((len(P.b),1)))
#ax, _, _ = F.plot()
#plt.show

P.A[:,:1],P.A[:,1:]

(array([[-0.],
        [-1.],
        [-0.],
        [-0.],
        [-0.],
        [ 1.]]),
 array([[-0., -1.],
        [-0., -0.],
        [-1., -0.],
        [-0.,  1.],
        [ 1., -0.],
        [-0., -0.]]))

In [ ]:
# Check proportional scalebility of sets

sigP = 0.5 * P

ax, _, _ = sigP.plot()
plt.show() 


# Casadi Tests

In [4]:
import numpy as np
import casadi as ca 
from pycvxset import Polytope

In [5]:
N = 30
n = 3
m = 2
op = ca.Opti()

x_opti =op.variable(n)
op.set_initial(x_opti,[1,2,3])
print(x_opti)

Q = ca.MX(np.array([[1,0,0],[0,2,0],[0,0,3]]))

# A = np.random(n,n)
# B = np.random(n.m)

x = ca.MX.sym('x',n,1,N)
x[0]=ca.MX(np.array([1,2,3]))
u = ca.MX.sym('u',m,1,N)
print(Q)
print(Q@x[0])
# constraint = []

# for i in range(N):

#     constraint.append(x[i+1]==A @ x[i] + B @ u[i])
#     ca.qpsol()



opti0_x_1

[[1, 0, 0], 
 [0, 2, 0], 
 [0, 0, 3]]
mac(
[[1, 0, 0], 
 [0, 2, 0], 
 [0, 0, 3]],[1, 2, 3],zeros(3x1))


## Casadi - Solving a NLP with Opti

In [ ]:
opti = ca.Opti()

x = opti.variable()
y = opti.variable()

opti.minimize((y-x**2)**2)
opti.subject_to(x**2+y**2==1)
opti.subject_to(x+y>1)

opti.solver('ipopt')

sol = opti.solve()

print(sol.value(x))
print(sol.value(y))

## Casadi - Solving a NLP with MX

In [15]:
x = ca.MX.sym('x')
y = ca.MX.sym('y')
F = ca.MX.sym("F")
G = ca.MX.sym("G")
f = (y-x**2)**2
g = ca.vertcat(x**2+y**2,x+y)
nlp = {'x':ca.vertcat(x,y), 'f':f, 'g':g}
s = ca.nlpsol('s', 'ipopt', nlp)
r = s(lbg=ca.vertcat(1,1), ubg=ca.vertcat(1,np.abs(-np.inf)))
x_opt = r['x']
print(x_opt)
z = ca.vertcat(x,y)
Const = ca.horzcat(F,G)
Const@z
g = ca.vertcat(g,Const@z)
const_value = []
h = np.array([[1,1,1],[2,2,2],[3,3,3]])
H = ca.MX(h)
print(H)

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        2
Number of nonzeros in inequality constraint Jacobian.:        2
Number of nonzeros in Lagrangian Hessian.............:        3

Total number of variables............................:        2
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        1
Total number of inequality constraints...............:        1
        inequality constraints with only lower bounds:        1
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 1.00e+00 1.00e+00  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

## Transforming a MX object into a numpy array - need to pass by DM object

In [20]:
print(H)
H_2 = H.to_DM()
H_test = np.zeros(H.shape)
for i in range(H_2.shape[0]):
    for j in range(H_2.shape[1]):
        H_test[i,j] = H_2[i,j]
H_test


[[1, 1, 1], 
 [2, 2, 2], 
 [3, 3, 3]]


array([[1., 1., 1.],
       [2., 2., 2.],
       [3., 3., 3.]])

# CVXPY LMI Solver

In [ ]:
import cvxpy as cp
import numpy as np

import os
import mosek

os.environ["MOSEKLM_LICENSE_FILE"] = "27007@192.168.100.118"
mosek.Env().getversion()
mosek.Env().putlicensepath("27007@192.168.100.118")
mosek.Env().expirylicenses()

# Data
A = np.array([[0, 1], [-1, -2]])
n = 2

# Variables
P = cp.Variable((n, n), symmetric=True)

tol = cp.Parameter(1,"tol")

# Constraints
# P > 0 and A.T*P + P*A < 0 (using a small epsilon for strict inequality)
constraints = [P >> tol * np.eye(n), 
               A.T @ P + P @ A << -tol * np.eye(n)]

# Problem
prob = cp.Problem(cp.Minimize(0), constraints)
#prob.solve()

#print("Status:", prob.status)
#print("P:\n", P.value)


In [ ]:
tol.value = [1e-6]
prob.solve()
print("Status:", prob.status)
print("P:\n", P.value)

In [ ]:
import scipy as sc
l=cp.Parameter(1,"lambda")
print(l)
l.shape

A = np.array([[1,2],[2,1]]).transpose()
print(np.linalg.inv(A))

In [ ]:
rho = np.array([[.9, .1, 2]])

if np.isnan(rho).any():
    print("Hello")

np.isnan(rho).any()

In [ ]:
import numpy as np
A=np.array([[[1, 2],[3, 4]],
            [[5, 6],[7, 8]],
            [[9, 10],[11, 12]]])
len(A.shape)
A_trans = np.copy(A).T

B=np.array([[1, 2],[3, 4]])
len(B.shape)

if len(A.shape)>2:
    v=A.shape
else:
    v=1


if len(B.shape)==2:
    B = B[np.newaxis, :, :]

b = np.array([1, 0])

B @ b

# Example of A[0]*a[0] + A[1]*a[1] + A[2]*a[2] ... where a is a vector of parameters and the first is always 1

a = np.array([0, 1, 0],).T
A = A.transpose(1, 2, 0)

A_trans @ a, A @ a, A[:, :, 0]


## pagemtimes() subsitution

In [ ]:
import numpy as np







